### Loading FIles

In [ ]:
# Import library
from langchain_community.document_loaders import PyPDFLoader

# Create a document loader for rag_paper.pdf
loader = PyPDFLoader('rag_paper.pdf')

# Load the document
data = loader.load()
print(data[0])

In [ ]:
# Create a document loader for unstructured HTML
loader = UnstructuredHTMLLoader('datacamp-blog.html')

# Load the document
data = loader.load()

# Print the first document's content
print(data[0].page_content)

# Print the first document's metadata
print(data[0].metadata)

### Text Splitting embeddings, and vector storage

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('rag_paper.pdf')
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

chunks = splitter.split_documents(documents)

In [ ]:
# Splitting documents
print(chunks)
print([len(chunks.page_content) for chunk in chunks])

### Embedding and Storage

##### Embedding and storing the chunks
- Embed and store with: OpenAI and ChromaDB

In [ ]:
from langchain_openai import OpeAIEmbeddings
from langchain_chroma import Chroma

embedding_model = OpenAIEmbeddings(api_key = openai_api_key, 
                                   model='text-embedding-3-small')

vector_store = Chroma.from_documents(documents = chunks,
                                     embedding = embedding_model)



In [ ]:
text = '''RAG (retrieval augmented generation) is an advanced NLP model that combines retrieval mechanisms with generative capabilities. RAG aims to improve the accuracy and relevance of its outputs by grounding responses in precise, contextually appropriate data.'''

# Define a text splitter that splits on the '.' character
text_splitter = CharacterTextSplitter(
    separator='.',
    chunk_size=75,
    chunk_overlap=10
)
# Split the text using text_splitter
chunks = text_splitter.split_text(text)
print(chunks)
print([len(chunk) for chunk in chunks])

In [ ]:
loader = PyPDFLoader("rag_paper.pdf")
document = loader.load()

# Define a text splitter that splits recursively through the character list
text_splitter = RecursiveCharacterTextSplitter(
    separators=['\n', '.', ' ', ''],
    chunk_size=75,  
    chunk_overlap=10  
)

# Split the document using text_splitter
chunks = text_splitter.split_documents(document)
print(chunks)
print([len(chunk.page_content) for chunk in chunks])

In [ ]:
# Initialize the OpenAI embedding model
embedding_model = OpenAIEmbeddings(api_key="<OPENAI_API_TOKEN>", model='text-embedding-3-small')

# Create a Chroma vector store and embed the chunks
vector_store = Chroma.from_documents(documents=chunks, embedding=embedding_model)

### Building an LCEL (LAngChain Expression Language) retrieval chain

In [ ]:
# Instantiating a retriever from the vector store
vector_store = Chroma.from_documents(documents=chunks, embedding=embedding_model)
retriever = vector_store.as_retriever(
    search_type="similarity", 
    search_kwargs={"k": 2}
)

In [ ]:
# Creating a prompt template
from langchain.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_templates("""
Use the following pieces of context to answer the question at the end. If you don't know the answer, say you don't know.
If you don't know the answer, say you don't know. Always use all available data to answer the question. Do not make up an answer if you don't know it.
Context:{context}
Question: {question}
""")


In [ ]:
llm = OpenAI(model='gpt-4o-mini', api_key=openai_api_key, temperature=0)


In [ ]:
# Building an LCEL retrieval chain
from langchain_core.runnables import RunnablePassThrough
from langchain_core.outputs_parsers import StrOutputParser

chain = (
    {"context": retriever, "question": RunnablePassThrough()} 
    | prompt 
    | llm 
    | StrOutputParser()
)

In [ ]:
# Invoking the retrieval chain with a question

result = chain.invoke({"question":"What are the key findings or results presented in the paper?"})
print(result)

### Exercise
1. Creating the retrieval prompt
2. Building the retrieval chain

In [ ]:
prompt = """
Use the only the context provided to answer the following question. If you don't know the answer, reply that you are unsure.
Context: {context}
Question: {question}
"""

# Convert the string into a chat prompt template
prompt_template = ChatPromptTemplate.from_template(prompt)

# Create an LCEL chain to test the prompt
chain = prompt_template | llm

# Invoke the chain on the inputs provided
print(chain.invoke({"context": "DataCamp's RAG course was created by Meri Nova and James Chapman!", "question": "Who created DataCamp's RAG course?"}))

In [ ]:
# Convert the vector store into a retriever
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 2})

# Create the LCEL retrieval chain
chain = (
    {"context": retriever, "question":  RunnablePassthrough()}
    | prompt_template
    | llm
)

# Invoke the chain
print(chain.invoke("Who are the authors?"))

In [ ]:
# Convert the vector store into a retriever
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 2})

# Create the LCEL retrieval chain
chain = (
    {"context": retriever, "question":  RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

# Invoke the chain
print(chain.invoke("Who are the authors?"))